In [1]:
# ============================================
# Imports
# ============================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import least_squares
import jax
import jax.numpy as jnp
import numpyro as pyro
import numpyro.distributions as dist

# Disable JAX warnings for cleaner output
import warnings
warnings.filterwarnings("ignore")


/Users/suheylatozan/Desktop/Movement Recovery Lab/hbmep/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import logging

import numpy as np
import jax
import jax.numpy as jnp
import numpyro as pyro
import numpyro.distributions as dist

from hbmep import functional as F, smooth_functional as SF
from hbmep.model import BaseModel
from hbmep.util import site

EPS = 1e-3


class HB(BaseModel):
    def __init__(self, *args, **kw):
        super(HB, self).__init__(*args, **kw)
        self.use_mixture = False

    def hb_rl(self, intensity, features, response=None, **kw):
        num_data = intensity.shape[0]
        num_features = np.max(features, axis=0) + 1

        # Mask missing observations
        mask_obs = True
        if response is not None:
            mask_obs = np.invert(np.isnan(response))

        # Hyper-priors
        a_loc = pyro.sample(
            site.a.log, dist.TruncatedNormal(5., 10., low=0)
        )
        a_scale = pyro.sample(site.a.scale, dist.HalfNormal(10.))
        b_scale = pyro.sample(site.b.scale, dist.HalfNormal(10.))
        h_scale = pyro.sample(site.h.scale, dist.HalfNormal(50.))

        g_scale = pyro.sample(site.g.scale, dist.HalfNormal(5.))
        v_scale = pyro.sample(site.v.scale, dist.HalfNormal(1.))

        c1_scale = pyro.sample(site.c1.scale, dist.HalfNormal(5.))
        c2_scale = pyro.sample(site.c2.scale, dist.HalfNormal(.5))

        # Priors
        with pyro.plate(site.num_response, self.num_response):
            with pyro.plate_stack(
                site.num_features, num_features, rightmost_dim=-2
            ):
                a = pyro.sample(
                    site.a, dist.TruncatedNormal(a_loc, a_scale, low=0)
                )

                b_raw = pyro.sample(site.b.raw, dist.HalfNormal(1))
                b = pyro.deterministic(site.b, b_scale * b_raw)

                g_raw = pyro.sample(site.g.raw, dist.HalfNormal(1))
                g = pyro.deterministic(site.g, g_scale * g_raw)

                h_raw = pyro.sample(site.h.raw, dist.HalfNormal(1))
                h = pyro.deterministic(site.h, h_scale * h_raw)

                v_raw = pyro.sample(site.v.raw, dist.HalfNormal(1))
                v = pyro.deterministic(site.v, v_scale * v_raw)

                c1_raw = pyro.sample(site.c1.raw, dist.HalfNormal(1))
                c1 = pyro.deterministic(site.c1, c1_scale * c1_raw)

                c2_raw = pyro.sample(site.c2.raw, dist.HalfNormal(1))
                c2 = pyro.deterministic(site.c2, c2_scale * c2_raw)

        # Outlier probability
        if self.use_mixture:
            q = pyro.sample(site.outlier_prob, dist.Uniform(0., 0.01))

        # Observation model
        with pyro.handlers.mask(mask=mask_obs):
            with pyro.plate(site.num_response, self.num_response):
                with pyro.plate(site.num_data, num_data):
                    mu = SF.rectified_logistic(
                        intensity,
                        a[*features.T],
                        b[*features.T],
                        g[*features.T],
                        h[*features.T],
                        v[*features.T],
                        EPS
                    )
                    alpha, beta = self.gamma_likelihood(
                        mu, 
                        c1[*features.T],
                        c2[*features.T],
                    )
                    pyro.deterministic(site.mu, mu)

                    # Mixture distribution
                    if self.use_mixture:
                        mixing_distribution = dist.Categorical(
                            probs=jnp.stack([1 - q, q], axis=-1)
                        )
                        component_distributions=[
                            dist.Gamma(concentration=alpha, rate=beta),
                            dist.HalfNormal(
                                scale=(g[*features.T] + h[*features.T])
                            )
                        ]
                        Mixture = dist.MixtureGeneral(
                            mixing_distribution=mixing_distribution,
                            component_distributions=component_distributions
                        )

                    # Observations
                    y_ = pyro.sample(
                        site.obs,
                        (
                            Mixture if self.use_mixture
                            else dist.Gamma(concentration=alpha, rate=beta)
                        ),
                        obs=response
                    )


In [4]:
csv_path = "/Users/suheylatozan/Desktop/Movement Recovery Lab/sc_ramp.csv"
df = pd.read_csv(csv_path)
req_cols = ["sc_current", "FCR", "participant", "recr_curve"]
df = df.dropna(subset=req_cols).copy()

group_cols = ["participant", "recr_curve"]


In [5]:
#Run hbMEP Bayesian Model

model = HB()
model.intensity = "sc_current"
model.features = group_cols
model.response = ["FCR"]
model._model = model.hb_rl
model.use_mixture = True
model.mcmc_params = dict(num_chains=2, num_warmup=500, num_samples=250) # Only way to determine is by analyzing convergence diagnostics. 

# Encode and run
df_enc, enc = model.load(df)
mcmc, posterior = model.run(df=df_enc)
pred_df = model.make_prediction_dataset(df=df_enc, num_points=200)
predictive = model.predict(df=pred_df, posterior=posterior, num_samples=500, return_sites=[site.mu])
pred_df["mu_post_mean"] = np.asarray(predictive[site.mu]).mean(axis=0)


Running chain 1: 100%|██████████| 750/750 [10:39<00:00,  1.17it/s]


In [6]:
mcmc.print_summary()


                   mean       std    median      5.0%     95.0%     n_eff     r_hat
     a[0,0,0]      0.47      0.12      0.48      0.28      0.64    196.11      1.01
     a[0,1,0]      2.13      0.12      2.16      2.05      2.31     90.87      1.01
     a[0,2,0]      0.36      0.16      0.36      0.06      0.58    538.01      1.00
     a[0,3,0]      1.21      0.15      1.22      1.02      1.44    168.54      1.01
     a[0,4,0]      1.68      0.25      1.69      1.28      1.98     76.14      1.04
     a[0,5,0]      1.97      0.28      2.08      1.40      2.21    133.99      1.01
     a[0,6,0]      0.61      0.20      0.66      0.30      0.96    130.79      1.00
     a[0,7,0]      0.53      0.05      0.53      0.45      0.62    464.37      1.00
     a[0,8,0]      0.79      0.22      0.82      0.35      1.06    235.54      1.00
     a[1,0,0]      1.13      0.14      1.15      0.99      1.31     65.63      1.01
     a[1,1,0]      0.83      0.09      0.83      0.69      0.97    489.11  